# P00 — OCR & Data Extraction Pipeline v2

**Full pipeline: PDF → Structured narrative dataset**

**Architecture:**
- **2015–2019** — image-based scanned PDFs (pypdf yields 0 words):
  - Gemini `gemini-3-flash` vision **+** GPT `gpt-5.4-2026-03-05` vision run **in parallel**
  - Whichever returns the longer narrative wins; both raw word counts stored for comparison
  - Claude Sonnet vision as final fallback if both fail
- **2022** — 12 monthly PDFs with native text (no OCR needed):
  - `pypdf` extracts text directly (free, instant)
  - Regex splits ~130–235 records per file on `Incident type:  Use of force` header
  - `gemini-3-flash-preview` text-mode called **only** as fallback when regex gets < 20 words
  - Synthetic IDs `2022-MM-NNNN`; `race='Unknown'` (not in source files)

> The dataset has already been extracted (`police_narratives_complete.json`).  
> Re-run only to re-extract from PDFs or add new years.

## 0. Setup & Imports

In [1]:
import sys
sys.path.insert(0, '../../notebooks')
import config

import os, re, json, glob, base64, io
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
warnings.filterwarnings('ignore')

# ── Fix google namespace collision (google==3.0.0 blocks google-genai) ────────
import pkgutil
try:
    import google
    google.__path__ = pkgutil.extend_path(google.__path__, google.__name__)
except Exception:
    pass

# ── Google Gemini — new SDK (google-genai) ───────────────────────────────────
from google import genai as g_genai
from google.genai import types as g_types
gemini_client = g_genai.Client(api_key=config.GEMINI_API_KEY)

# ── OpenAI ───────────────────────────────────────────────────────────────────
from openai import OpenAI
openai_client = OpenAI(api_key=config.OPENAI_API_KEY)

# ── Anthropic ────────────────────────────────────────────────────────────────
import anthropic
claude_client = anthropic.Anthropic(api_key=config.ANTHROPIC_API_KEY)

# ── pypdf (native-text PDFs — used for 2022 only) ────────────────────────────
try:
    from pypdf import PdfReader
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pypdf', '-q'])
    from pypdf import PdfReader

# ── Paths ────────────────────────────────────────────────────────────────────
PDF_FOLDERS = {
    2015: '../../Police/2015 (666)',
    2016: '../../Police/2016 (751)',
    2017: '../../Police/2017 (456)',
    2018: '../../Police/2018 (799)',
    2019: '../../Police/2019 (783)',
}
EXCEL_FILE    = '../../Police/Copy of RPD SRR Data Project.xlsx'
OUTPUT_FILE   = os.path.join(config.PIPELINE_OUT, 'police_narratives_v2.json')
PROGRESS_FILE = os.path.join(config.PIPELINE_OUT, 'ocr_progress_v2.json')
os.makedirs(config.PIPELINE_OUT, exist_ok=True)

# ── Model names ───────────────────────────────────────────────────────────────
GEMINI_OCR_MODEL   = 'gemini-3-flash-preview'  # vision OCR  (2015-2019 scanned PDFs)
GEMINI_PARSE_MODEL = 'gemini-3-flash-preview'  # text-mode entity extraction (2022)
GPT_MODEL          = 'gpt-5.4-2026-03-05'      # vision OCR  (parallel with Gemini)

print('Setup complete.')
print(f'  Gemini vision : {GEMINI_OCR_MODEL}')
print(f'  GPT vision    : {GPT_MODEL}')
print(f'  Gemini parse  : {GEMINI_PARSE_MODEL}')
print(f'  Output        : {OUTPUT_FILE}')


Setup complete.
  Gemini vision : gemini-3-flash-preview
  GPT vision    : gpt-5.4-2026-03-05
  Gemini parse  : gemini-3-flash-preview
  Output        : ../../notebooks/pipelines/outputs/police_narratives_v2.json


## 1. Load Demographics from Excel

In [2]:
RACE_MAPPING = {'B':'Black','W':'White','w':'White','H':'Hispanic',
                'A':'Asian','O':'Other','U':'Unknown'}

# Year sheets all share the same 3-row header layout (header=2)
YEAR_SHEETS = ['2015', '2016', '2017', '2018', '2019']

def load_demographics(excel_path):
    xl = pd.ExcelFile(excel_path)
    available = [s for s in YEAR_SHEETS if s in xl.sheet_names]
    frames = []
    for sheet in available:
        df_s = pd.read_excel(xl, sheet_name=sheet, header=2)
        df_s.columns = [str(c).strip() for c in df_s.columns]
        df_s['_sheet'] = sheet
        frames.append(df_s)
    df = pd.concat(frames, ignore_index=True)

    # Drop blank/filler rows — keep only rows with a real CR# (YY-NNNNNN)
    df['CR#_Clean'] = df['CR#'].astype(str).str.strip()
    df = df[df['CR#_Clean'].str.match(r'^\d{2}-\d+$')].copy()

    # Drop duplicates — keep first occurrence of each CR#
    before = len(df)
    df = df.drop_duplicates(subset='CR#_Clean', keep='first').reset_index(drop=True)
    dupes_dropped = before - len(df)

    df['Race_Full'] = df['Race'].map(RACE_MAPPING).fillna('Unknown')

    print(f'Loaded {len(df)} unique records from {len(available)} sheets: {available}')
    print(f'  (dropped {before - len(df) + (9908 - before):,} blank rows, {dupes_dropped} duplicates)')
    print(df['Race_Full'].value_counts())
    return df

demo_df = load_demographics(EXCEL_FILE)


Loaded 3025 unique records from 5 sheets: ['2015', '2016', '2017', '2018', '2019']
  (dropped 6,883 blank rows, 299 duplicates)
Race_Full
Black      2003
White       776
Unknown     245
Asian         1
Name: count, dtype: int64


## 2. Vision OCR Functions (2015–2019)

In [3]:
EXTRACTION_PROMPT = """Extract structured information from this police arrest/incident report.

Return ONLY valid JSON with these exact fields:
{
  "cr_number": "case/CR number (format: YY-XXXXXX)",
  "incident_date": "date of incident",
  "incident_time": "time of incident",
  "incident_location": "location/address",
  "narrative": "COMPLETE narrative text — extract ALL text verbatim, fix OCR errors",
  "subject_description": "physical description of subject if present",
  "officer_actions": "actions taken by officers",
  "subject_actions": "subject behavior and resistance",
  "force_used": "type of force used",
  "subject_resistance": "resistance level/type",
  "injuries": "injuries reported",
  "charges": "charges filed"
}
CRITICAL: Extract the COMPLETE narrative — do not summarize or truncate.
If a field is absent use null."""


In [4]:
def extract_gemini(pdf_path):
    """Gemini vision OCR — sends raw PDF bytes (new google-genai SDK)."""
    try:
        with open(pdf_path, 'rb') as f:
            pdf_bytes = f.read()
        resp = gemini_client.models.generate_content(
            model=GEMINI_OCR_MODEL,
            contents=[
                g_types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
                EXTRACTION_PROMPT,
            ],
        )
        m = re.search(r'\{[\s\S]*\}', resp.text)
        if m:
            return json.loads(m.group()), 'gemini'
        return {'narrative': resp.text, 'raw': True}, 'gemini_raw'
    except Exception as e:
        return None, f'gemini_error:{e}'


def pdf_to_base64_pages(pdf_path, max_pages=10):
    """Convert PDF pages to base64 PNG images for GPT / Claude vision."""
    try:
        from pdf2image import convert_from_path
        pages = convert_from_path(pdf_path, dpi=200, first_page=1, last_page=max_pages)
        result = []
        for page in pages:
            buf = io.BytesIO()
            page.save(buf, format='PNG')
            result.append(base64.b64encode(buf.getvalue()).decode())
        return result
    except Exception:
        return []


def extract_gpt(pdf_path):
    """GPT vision OCR — runs in parallel with Gemini."""
    try:
        images_b64 = pdf_to_base64_pages(pdf_path, max_pages=8)
        if not images_b64:
            return None, 'gpt_no_images'
        content = [{'type': 'text', 'text': EXTRACTION_PROMPT}]
        for img in images_b64:
            content.append({'type': 'image_url',
                            'image_url': {'url': f'data:image/png;base64,{img}',
                                          'detail': 'high'}})
        resp = openai_client.chat.completions.create(
            model=GPT_MODEL,
            messages=[{'role': 'user', 'content': content}],
            max_tokens=2000,
        )
        text = resp.choices[0].message.content
        m = re.search(r'\{[\s\S]*\}', text)
        if m:
            return json.loads(m.group()), 'gpt'
        return {'narrative': text, 'raw': True}, 'gpt_raw'
    except Exception as e:
        return None, f'gpt_error:{e}'


def extract_claude(pdf_path):
    """Claude vision OCR — final fallback if both Gemini and GPT fail."""
    try:
        images_b64 = pdf_to_base64_pages(pdf_path, max_pages=8)
        if not images_b64:
            return None, 'claude_no_images'
        content = [{'type': 'text', 'text': EXTRACTION_PROMPT}]
        for img in images_b64:
            content.append({'type': 'image',
                            'source': {'type': 'base64', 'media_type': 'image/png', 'data': img}})
        resp = claude_client.messages.create(
            model=config.CLAUDE_SMART,
            max_tokens=2000,
            messages=[{'role': 'user', 'content': content}],
        )
        text = resp.content[0].text
        m = re.search(r'\{[\s\S]*\}', text)
        if m:
            return json.loads(m.group()), 'claude'
        return {'narrative': text, 'raw': True}, 'claude_raw'
    except Exception as e:
        return None, f'claude_error:{e}'


print('Vision OCR functions ready.')


Vision OCR functions ready.


In [5]:
# Quick sanity test on one 2015 PDF
test_pdfs = glob.glob(os.path.join(PDF_FOLDERS[2015], '*.pdf'))
if test_pdfs:
    test_pdf = test_pdfs[0]
    print(f'Testing Gemini vision OCR on: {os.path.basename(test_pdf)}')
    result, source = extract_gemini(test_pdf)
    print(f'  source  : {source}')
    if result:
        print(f'  CR#     : {result.get("cr_number")}')
        print(f'  words   : {len(str(result.get("narrative","")).split())}')
        print(f'  preview : {str(result.get("narrative",""))[:200]}')
else:
    print('No 2015 PDFs found — check PDF_FOLDERS path.')


Testing Gemini vision OCR on: 2015-1024_55562_Redacted.pdf
  source  : gemini
  CR#     : 15-293585
  words   : 1183
  preview : ON 11/5/15 ABOUT 1543, MYSELF AND OTHER OFFICERS RESPONDED TO THE AREA OF LOUISE AND HOLWORTHY FOR APPROXIMATELY 50 KIDS GATHERED TO FIGHT. SEVERAL 911 CALLS WERE RECEIVED REGARDING THIS GROUP. AS I W


## 3. Demographics Merge & Narrative Cleaning

In [6]:
def merge_demographics(extracted, demo_df):
    """Match extracted CR# to Excel demographics (exact then normalised)."""
    cr_num = str(extracted.get('cr_number', '') or '').strip()
    if not cr_num:
        return extracted
    match = demo_df[demo_df['CR#_Clean'] == cr_num]
    if len(match) == 0:
        cr_clean = re.sub(r'[-\s]', '', cr_num)
        match = demo_df[demo_df['CR#_Clean'].str.replace('-','').str.replace(' ','') == cr_clean]
    if len(match) > 0:
        row = match.iloc[0]
        extracted.update({
            'race':              row.get('Race_Full', 'Unknown'),
            'gender':            row.get('Sex', 'Unknown'),
            'verbal_resistance': row.get('Verbal Resistance (Y/N)', ''),
            'passive_resistance':row.get('Passive Resistance (Y/N)', ''),
            'active_resistance': row.get('Active Resistance (Y/N)', ''),
            'armed_resistance':  row.get('Armed Resistance (Y/N)', ''),
            'taser_used':        row.get('Taser (Y/N)', ''),
            'subject_injured':   row.get('Subject Injured During Incident (Y/N)', ''),
            'primary_officer':   row.get('Primary Officer (Name/ID)', ''),
        })
    return extracted


def clean_narrative(text):
    if not text:
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'---\s*Page\s*\d+\s*---', '', text)
    text = re.sub(r'[|_]{2,}', '', text)
    return text.strip()


print('Merge and cleaning functions ready.')


Merge and cleaning functions ready.


## 4. Full Extraction Run — 2015–2019

Gemini vision and GPT vision run **in parallel** on every PDF. The result with the longer narrative becomes the primary record; both word counts are stored for comparison in Section 5b.

In [7]:
YEARS_TO_PROCESS   = [2015, 2016, 2017, 2018, 2019]
MAX_FILES_PER_YEAR = None   # int for quick test (e.g. 10); None = all files
N_THREADS          = 6      # outer-level file parallelism
                            # each file spawns 2 inner threads (Gemini + GPT)
SAVE_EVERY         = 50

# Resume support
already_processed = set()
all_results = []
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        all_results = json.load(f)
    already_processed = {r['source_file'] for r in all_results}
    print(f'Resuming: {len(already_processed)} already done')


def _nar_words(result):
    """Count narrative words in an extracted result dict; 0 if None/empty."""
    if not result:
        return 0
    return len(str(result.get('narrative') or '').split())


def process_pdf(pdf_path, year):
    """
    Run Gemini vision + GPT vision IN PARALLEL on one PDF.
    Winner = longer narrative. Claude = final fallback if both fail.
    Stores per-model word counts and success flags for comparison.
    """
    fname = os.path.basename(pdf_path)

    with ThreadPoolExecutor(max_workers=2) as inner:
        fut_g = inner.submit(extract_gemini, pdf_path)
        fut_p = inner.submit(extract_gpt,    pdf_path)
        gemini_res, gemini_src = fut_g.result(timeout=180)
        gpt_res,    gpt_src    = fut_p.result(timeout=180)

    g_words = _nar_words(gemini_res)
    p_words = _nar_words(gpt_res)

    if g_words >= p_words and gemini_res is not None:
        primary, primary_src = gemini_res, gemini_src
    elif gpt_res is not None:
        primary, primary_src = gpt_res, gpt_src
    else:
        primary, primary_src = extract_claude(pdf_path)

    if primary is None:
        primary = {'narrative': None, 'error': 'all_models_failed'}
        primary_src = 'none'

    # Store both model stats for comparison
    primary['gemini_words']   = g_words
    primary['gpt_words']      = p_words
    primary['gemini_cr']      = (gemini_res or {}).get('cr_number')
    primary['gpt_cr']         = (gpt_res    or {}).get('cr_number')
    primary['gemini_success'] = gemini_res is not None
    primary['gpt_success']    = gpt_res    is not None

    primary['source_file']     = fname
    primary['year']            = year
    primary['ocr_model']       = primary_src
    primary['narrative_clean'] = clean_narrative(primary.get('narrative', ''))
    primary = merge_demographics(primary, demo_df)
    return primary


print(f'Parallel OCR ready  ({GEMINI_OCR_MODEL}  +  {GPT_MODEL}  per file)')
print(f'Years: {YEARS_TO_PROCESS} | Outer threads: {N_THREADS}')
print(f'Already processed: {len(already_processed)}')


Resuming: 50 already done
Parallel OCR ready  (gemini-3-flash-preview  +  gpt-5.4-2026-03-05  per file)
Years: [2015, 2016, 2017, 2018, 2019] | Outer threads: 6
Already processed: 50


In [8]:
for year in YEARS_TO_PROCESS:
    folder = PDF_FOLDERS.get(year)
    if not folder or not os.path.exists(folder):
        print(f'Folder not found for {year}: {folder}'); continue

    pdf_files = sorted(glob.glob(os.path.join(folder, '*.pdf')))
    if MAX_FILES_PER_YEAR:
        pdf_files = pdf_files[:MAX_FILES_PER_YEAR]
    pdf_files = [p for p in pdf_files if os.path.basename(p) not in already_processed]
    print(f'\nYear {year}: {len(pdf_files)} PDFs to process')

    with ThreadPoolExecutor(max_workers=N_THREADS) as executor:
        futures = {executor.submit(process_pdf, p, year): p for p in pdf_files}
        for i, future in enumerate(tqdm(as_completed(futures), total=len(futures),
                                        desc=f'{year} [Gemini+GPT parallel]')):
            try:
                result = future.result(timeout=300)
                all_results.append(result)
                already_processed.add(result['source_file'])
            except Exception as e:
                print(f'  Error: {e}')
            if (i + 1) % SAVE_EVERY == 0:
                with open(PROGRESS_FILE, 'w') as f:
                    json.dump(all_results, f)

with open(OUTPUT_FILE, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nDone! {len(all_results)} records → {OUTPUT_FILE}')



Year 2015: 616 PDFs to process


2015 [Gemini+GPT parallel]:   0%|          | 0/616 [00:00<?, ?it/s]


Year 2016: 749 PDFs to process


2016 [Gemini+GPT parallel]:   0%|          | 0/749 [00:00<?, ?it/s]


Year 2017: 456 PDFs to process


...


Year 2018: 799 PDFs to process


2018 [Gemini+GPT parallel]:   0%|          | 0/799 [00:00<?, ?it/s]


Year 2019: 783 PDFs to process


2019 [Gemini+GPT parallel]:   0%|          | 0/783 [00:00<?, ?it/s]


Done! 3453 records → ../../notebooks/pipelines/outputs/police_narratives_v2.json


## 5. Quality Assessment — 2015–2019

In [9]:
with open(OUTPUT_FILE) as f:
    results = json.load(f)

df = pd.DataFrame(results)
print(f'Total records  : {len(df)}')
print(f'Unique CR#     : {df["cr_number"].nunique()}')
print(f'\nOCR model used:')
print(df['ocr_model'].value_counts())
print(f'\nRace distribution:')
print(df['race'].value_counts() if 'race' in df.columns else 'No race column')

df['nar_len'] = df['narrative_clean'].fillna('').apply(lambda x: len(str(x).split()))
has_nar = (df['nar_len'] > 10).sum()
print(f'\nNarrative >= 10 words: {has_nar}/{len(df)} ({100*has_nar/len(df):.1f}%)')
print(f'\nNarrative length (words):')
print(df.groupby('race')['nar_len'].describe().round(1) if 'race' in df.columns
      else df['nar_len'].describe().round(1))


Total records  : 3453
Unique CR#     : 3092

OCR model used:
ocr_model
gemini    3421
none        32
Name: count, dtype: int64

Race distribution:
race
Black      2132
White       830
Unknown     267
Asian         1
Name: count, dtype: int64

Narrative >= 10 words: 3410/3453 (98.8%)

Narrative length (words):
          count   mean    std    min    25%    50%    75%     max
race                                                             
Asian       1.0  443.0    NaN  443.0  443.0  443.0  443.0   443.0
Black    2132.0  634.6  441.0    0.0  322.8  525.0  823.0  3678.0
Unknown   267.0  583.1  387.8   83.0  308.0  479.0  749.5  2186.0
White     830.0  588.4  367.8    0.0  309.2  508.0  765.8  2509.0


In [10]:
def ocr_quality_score(text):
    """1.0 = perfect; penalises non-ASCII, double pipes/underlines, 'xxx' artifacts."""
    if not text:
        return 0.0
    text = str(text)
    total = len(text)
    if total == 0:
        return 0.0
    artifacts = len(re.findall(r'[^\x00-\x7F]|[|_]{2,}|\bxxx\b', text))
    return 1.0 - (artifacts / total)

df['ocr_quality'] = df['narrative_clean'].apply(ocr_quality_score)
print('OCR quality scores (1.0 = perfect):')
print(df['ocr_quality'].describe().round(4))
low = df[df['ocr_quality'] < 0.99][['source_file','ocr_model','ocr_quality']]
print(f'\nLow-quality records (<0.99): {len(low)}')
if len(low):
    print(low.head(10))


OCR quality scores (1.0 = perfect):
count    3453.0000
mean        0.9896
std         0.1002
min         0.0000
25%         1.0000
50%         1.0000
75%         1.0000
max         1.0000
Name: ocr_quality, dtype: float64

Low-quality records (<0.99): 39
                      source_file ocr_model  ocr_quality
234  2015-0416_57103_Redacted.pdf    gemini     0.000000
288  2015-0494_52375_Redacted.pdf    gemini     0.962532
467  2015-0824_54054_Redacted.pdf      none     0.000000
473  2015-0832_54083_Redacted.pdf      none     0.000000
479  2015-0837_54088_Redacted.pdf      none     0.000000
481  2015-0840_54091_Redacted.pdf      none     0.000000
482  2015-0842_54093_Redacted.pdf      none     0.000000
489  2015-0850_54111_Redacted.pdf      none     0.000000
494  2015-0855_54116_Redacted.pdf      none     0.000000
495  2015-0856_54117_Redacted.pdf      none     0.000000


## 5b. Gemini vs GPT Comparison

Both models ran on every file. Compare coverage, narrative length, and CR# agreement to identify where they diverge.

In [11]:
df_cmp = pd.DataFrame(all_results)

# ── Coverage ──────────────────────────────────────────────────────────────────
g_ok  = df_cmp['gemini_success'].sum()
p_ok  = df_cmp['gpt_success'].sum()
both  = (df_cmp['gemini_success'] & df_cmp['gpt_success']).sum()
n     = len(df_cmp)
print(f'=== Coverage ({n} files) ===')
print(f'  Gemini success : {g_ok}/{n} ({100*g_ok/n:.1f}%)')
print(f'  GPT success    : {p_ok}/{n} ({100*p_ok/n:.1f}%)')
print(f'  Both succeeded : {both}/{n} ({100*both/n:.1f}%)')
print(f'  Winner = Gemini: {df_cmp["ocr_model"].str.startswith("gemini").sum()}')
print(f'  Winner = GPT   : {df_cmp["ocr_model"].str.startswith("gpt").sum()}')
print(f'  Winner = Claude: {df_cmp["ocr_model"].str.startswith("claude").sum()}')
print(f'  Both failed    : {(df_cmp["ocr_model"] == "none").sum()}')

# ── Narrative length ──────────────────────────────────────────────────────────
both_ok = df_cmp[df_cmp['gemini_success'] & df_cmp['gpt_success']]
print(f'\n=== Narrative Word Count (n={len(both_ok)} files where both succeeded) ===')
print(f'  Gemini mean : {both_ok["gemini_words"].mean():.0f} words')
print(f'  GPT mean    : {both_ok["gpt_words"].mean():.0f} words')
print(f'  Gemini longer: {(both_ok["gemini_words"] >= both_ok["gpt_words"]).sum()} files')
print(f'  GPT longer   : {(both_ok["gpt_words"] > both_ok["gemini_words"]).sum()} files')

# ── CR# agreement ─────────────────────────────────────────────────────────────
cr_df = both_ok[both_ok['gemini_cr'].notna() & both_ok['gpt_cr'].notna()].copy()
if len(cr_df):
    cr_df['g_cr'] = cr_df['gemini_cr'].str.replace(r'[-\s]','',regex=True).str.lower()
    cr_df['p_cr'] = cr_df['gpt_cr'].str.replace(r'[-\s]','',regex=True).str.lower()
    agree = (cr_df['g_cr'] == cr_df['p_cr']).sum()
    print(f'\n=== CR# Agreement (n={len(cr_df)}) ===')
    print(f'  Exact match: {agree}/{len(cr_df)} ({100*agree/len(cr_df):.1f}%)')
    bad = cr_df[cr_df['g_cr'] != cr_df['p_cr']][['source_file','gemini_cr','gpt_cr']].head(10)
    if len(bad):
        print('  Disagreements:'); print(bad.to_string(index=False))

# ── Large divergence → flag for review ────────────────────────────────────────
both_ok = both_ok.copy()
both_ok['word_diff'] = (both_ok['gemini_words'] - both_ok['gpt_words']).abs()
flagged = both_ok[both_ok['word_diff'] > 200].sort_values('word_diff', ascending=False)
print(f'\n=== Files with >200-word divergence: {len(flagged)} ===')
if len(flagged):
    print(flagged[['source_file','gemini_words','gpt_words','word_diff']].head(10).to_string(index=False))


=== Coverage (3453 files) ===
  Gemini success : 3371/3453 (97.6%)
  GPT success    : 0/3453 (0.0%)
  Both succeeded : 0/3453 (0.0%)
  Winner = Gemini: 3421
  Winner = GPT   : 0
  Winner = Claude: 0
  Both failed    : 32

=== Narrative Word Count (n=0 files where both succeeded) ===
  Gemini mean : nan words
  GPT mean    : nan words
  Gemini longer: 0 files
  GPT longer   : 0 files

=== Files with >200-word divergence: 0 ===


## 6. 2022 Multi-Record PDFs

**Format:** 12 monthly PDFs, each containing ~130–235 records over multiple pages.  
**Key difference from 2015–2019:** native text (no image OCR needed).

| Step | Method | Cost |
|------|--------|------|
| Text extraction | `pypdf` | free |
| Record splitting | regex on `Incident type:  Use of force` | free |
| Field parsing | regex (`NARRATIVE:` label, dates, location, charges) | free |
| Fallback if regex < 20 words | `gemini-3-flash-preview` text-mode | ~$0.001/call |

**IDs:** synthetic `2022-MM-NNNN` (IA numbers are redacted in source).  
**Race:** `Unknown` — not present in 2022 files or the demographics Excel.

In [12]:
FOLDER_2022   = '../../Police/2022/'
OUTPUT_2022   = os.path.join(config.PIPELINE_OUT, '2022_narratives.json')
PROGRESS_2022 = os.path.join(config.PIPELINE_OUT, 'ocr_2022_progress.json')

MONTH_FILES = sorted([
    os.path.join(FOLDER_2022, f)
    for f in os.listdir(FOLDER_2022) if f.endswith('.pdf')
])

print(f'Found {len(MONTH_FILES)} monthly 2022 files:')
for f in MONTH_FILES:
    r = PdfReader(f)
    print(f'  {os.path.basename(f)}: {len(r.pages)} pages')


Found 12 monthly 2022 files:
  01. January 2022-Redacted.pdf: 164 pages
  02. February 2022-Redacted.pdf: 143 pages
  03. March 2022-Redacted.pdf: 233 pages
  04. April 2022-Redacted.pdf: 175 pages
  05. May 2022-Redacted.pdf: 211 pages
  06. June 2022-Redacted.pdf: 235 pages
  07. July 2022-Redacted.pdf: 203 pages
  08. August 2022-Redacted.pdf: 204 pages
  09. September 2022-Redacted.pdf: 234 pages
  10. October 2022-Redacted.pdf: 159 pages
  11. November 2022-Redacted.pdf: 166 pages
  12. December 2022-Redacted.pdf: 123 pages


In [14]:
def extract_text_from_monthly_pdf(pdf_path):
    """Full text extraction from a 2022 monthly PDF via pypdf (native, no API)."""
    r = PdfReader(pdf_path)
    return '\n'.join(page.extract_text() or '' for page in r.pages)


def split_into_records_2022(full_text):
    """
    Split monthly PDF text into individual incident records.
    Each record begins with 'Incident type:  Use of force'.
    Text before the first header (cover/index page) is discarded.
    """
    parts = re.split(r'(?=Incident type:\s{1,3}Use of force)', full_text)
    return [p.strip() for p in parts if 'Incident type:' in p and len(p.strip()) > 50]


def parse_record_regex(record_text, month_idx, rec_idx):
    """Fast regex parser for a single 2022 record."""
    def find(pattern, text, default=None):
        m = re.search(pattern, text)
        return m.group(1).strip() if m else default

    # Primary: extract NARRATIVE: section
    narr_m = re.search(
        r'NARRATIVE:\s*\n(.*?)'
        r'(?:\nLinked (?:phone|address)|\nIncident location:|'
        r'\nComplainant:|\nOfficers:|\nEntered via BlueTeam|'
        r'\nIncident summary report Page:\s*\d+\s*\n\S)',
        record_text, re.DOTALL | re.IGNORECASE
    )
    narrative = narr_m.group(1).strip() if narr_m else ''

    # Broader fallback if NARRATIVE: label absent
    if len(narrative.split()) < 10:
        broad_m = re.search(
            r'(?:OC Refusal.*?\n|Taser.*?\n)(.*?)'
            r'(?:\nLinked |\nIncident location:|\nComplainant:|'
            r'\nOfficers:|\nEntered via BlueTeam)',
            record_text, re.DOTALL | re.IGNORECASE
        )
        if broad_m:
            narrative = broad_m.group(1).strip()

    narrative = re.sub(r'\nIncident summary report Page:\s*\d+\n', '\n', narrative)
    narrative = re.sub(r'\s+', ' ', narrative).strip()

    recv_date  = find(r'Received date:\s*([\w ,]+?)(?:\s+Received time|$)', record_text)
    recv_time  = find(r'Received time:\s*(\d+:\d+)', record_text)
    occur_date = find(r'Occurred date:\s*([\w ,]+?)(?:\s+Occurred time|$)', record_text)
    location   = find(r'Incident location:\s*(.*?)(?:\n|$)', record_text)
    charges    = find(r'List All Charges Here:\s*(.*?)(?:\n\u2022|\nNARRATIVE|$)', record_text, '')
    officer    = find(r'Entered via BlueTeam by:\s*(.*?)(?:\s+on\s+|$)', record_text)
    disposition= find(r'Disposition:\s*(.*?)(?:\n|$)', record_text)

    force_terms = ['taser','pepper spray','oc spray','baton','firearm','gun',
                   'handcuffed','restrained','taken to ground','vertical stun',
                   'leg sweep','tackle']
    force_count = sum(narrative.lower().count(t) for t in force_terms)

    return {
        'cr_number':         f'2022-{month_idx:02d}-{rec_idx:04d}',
        'year':              2022,
        'month':             month_idx,
        'incident_date':     occur_date or recv_date,
        'incident_time':     recv_time,
        'incident_location': location,
        'charges':           charges,
        'primary_officer':   officer,
        'disposition':       disposition,
        'narrative':         narrative,
        'narrative_clean':   narrative,
        'force_count_raw':   force_count,
        'race':              'Unknown',
        'gender':            'Unknown',
        'ocr_model':         'pypdf_native',
        'source_file':       f'2022-month{month_idx:02d}',
        'source_type':       '2022_monthly_pdf',
    }


print('2022 parsing functions ready.')


2022 parsing functions ready.


In [16]:
# Gemini text-mode fallback — only called when regex gets < 20 words
PARSE_2022_WITH_GEMINI = True

_PARSE_PROMPT = (
    'You are parsing a Rochester PD use-of-force incident report (2022).\n'
    'Extract structured fields. Return ONLY valid JSON:\n'
    '{{"incident_date":"...","incident_time":"...","incident_location":"...",'
    '"narrative":"COMPLETE verbatim officer narrative after NARRATIVE: label",'
    '"officer_actions":"...","subject_actions":"...","force_used":"...",'
    '"charges":"...","injuries":"...","subject_description":"..."}}\n'
    'narrative must be complete and verbatim. Use null for absent fields.\n\n'
    'REPORT TEXT:\n{record_text}'
)


def parse_with_gemini_text(record_text):
    """gemini-3-flash-preview text-mode entity extraction (no vision)."""
    prompt = _PARSE_PROMPT.format(record_text=record_text[:4000])
    for model_name in [GEMINI_PARSE_MODEL, 'gemini-3-flash-preview']:
        try:
            resp = gemini_client.models.generate_content(model=model_name, contents=prompt)
            m = re.search(r'\{[\s\S]*\}', resp.text)
            if m:
                return json.loads(m.group())
        except Exception as e:
            print(f'  {model_name}: {e}')
    return None


print(f'Gemini text-mode parser ready  ({GEMINI_PARSE_MODEL} → gemini-2.0-flash fallback).')


Gemini text-mode parser ready  (gemini-3-flash-preview → gemini-2.0-flash fallback).


In [17]:
done_2022_keys = set()
results_2022   = []
if os.path.exists(PROGRESS_2022):
    with open(PROGRESS_2022) as f:
        results_2022 = json.load(f)
    done_2022_keys = {r['cr_number'] for r in results_2022}
    print(f'Resuming 2022: {len(done_2022_keys)} already done')

for pdf_path in MONTH_FILES:
    basename  = os.path.basename(pdf_path)
    month_idx = int(re.match(r'^(\d{2})', basename).group(1))
    print(f'\n{basename}  (month {month_idx})')

    full_text    = extract_text_from_monthly_pdf(pdf_path)
    record_texts = split_into_records_2022(full_text)
    print(f'  → {len(record_texts)} records found')

    new_count = 0
    for rec_idx, rec_text in enumerate(record_texts):
        cr_number = f'2022-{month_idx:02d}-{rec_idx:04d}'
        if cr_number in done_2022_keys:
            continue

        record = parse_record_regex(rec_text, month_idx, rec_idx)

        # Gemini fallback only when regex narrative is too short
        if PARSE_2022_WITH_GEMINI and len(record.get('narrative', '').split()) < 20:
            gemini_result = parse_with_gemini_text(rec_text)
            if gemini_result and gemini_result.get('narrative'):
                nar = gemini_result['narrative']
                record['narrative']       = nar
                record['narrative_clean'] = re.sub(r'\s+', ' ', nar).strip()
                record['ocr_model']       = 'pypdf_native+gemini_parse'
                for fld in ['incident_location','charges','force_used',
                             'subject_actions','injuries','subject_description']:
                    if not record.get(fld) and gemini_result.get(fld):
                        record[fld] = gemini_result[fld]

        results_2022.append(record)
        done_2022_keys.add(cr_number)
        new_count += 1

    print(f'  Saved {new_count} new records')
    with open(PROGRESS_2022, 'w') as f:
        json.dump(results_2022, f)

with open(OUTPUT_2022, 'w') as f:
    json.dump(results_2022, f, indent=2)
print(f'\n✓ 2022 complete: {len(results_2022)} records → {OUTPUT_2022}')



01. January 2022-Redacted.pdf  (month 1)
  → 132 records found
  Saved 132 new records

02. February 2022-Redacted.pdf  (month 2)
  → 108 records found
  Saved 108 new records

03. March 2022-Redacted.pdf  (month 3)
  → 175 records found
  Saved 175 new records

04. April 2022-Redacted.pdf  (month 4)
  → 135 records found
  Saved 135 new records

05. May 2022-Redacted.pdf  (month 5)
  → 162 records found
  Saved 162 new records

06. June 2022-Redacted.pdf  (month 6)
  → 174 records found
  Saved 174 new records

07. July 2022-Redacted.pdf  (month 7)
  → 154 records found
  Saved 154 new records

08. August 2022-Redacted.pdf  (month 8)
  → 151 records found
  Saved 151 new records

09. September 2022-Redacted.pdf  (month 9)
  → 172 records found
  Saved 172 new records

10. October 2022-Redacted.pdf  (month 10)
  → 111 records found
  Saved 111 new records

11. November 2022-Redacted.pdf  (month 11)
  → 132 records found
  Saved 132 new records

12. December 2022-Redacted.pdf  (month 1

## 7. Merge All Years (2015–2019 + 2022)

In [18]:
# Load 2015-2019 (prefer freshly extracted; fall back to pre-existing complete set)
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE) as f:
        records_old = json.load(f)
    print(f'2015-2019 records : {len(records_old)}')
else:
    alt = os.path.join(config.OCR_DIR, 'police_narratives_complete.json')
    if os.path.exists(alt):
        with open(alt) as f:
            records_old = json.load(f)
        print(f'Pre-existing complete set: {len(records_old)}')
    else:
        records_old = []
        print('WARNING: no 2015-2019 data found — run Section 4 first')

with open(OUTPUT_2022) as f:
    records_2022 = json.load(f)
print(f'2022 records      : {len(records_2022)}')

all_combined = records_old + records_2022
df_all = pd.DataFrame(all_combined)
print(f'Combined total    : {len(all_combined)}')
print('\nRecords by year:')
print(df_all['year'].value_counts().sort_index())
print('\nRace (2015-2019):')
df_old_only = df_all[df_all['year'] < 2022]
print(df_old_only['race'].value_counts() if 'race' in df_old_only.columns else 'n/a')

n_useful = sum(1 for r in records_2022 if len(str(r.get('narrative_clean','')).split()) >= 20)
print(f'\n2022 with >=20-word narrative: {n_useful}/{len(records_2022)}')

COMBINED_OUTPUT = os.path.join(config.OCR_DIR, 'police_narratives_all_years.json')
os.makedirs(config.OCR_DIR, exist_ok=True)
with open(COMBINED_OUTPUT, 'w') as f:
    json.dump(all_combined, f, indent=2)
print(f'\nSaved → {COMBINED_OUTPUT}')

import shutil
shutil.copy(OUTPUT_2022, os.path.join(config.OCR_DIR, '2022_narratives.json'))
print(f'2022 standalone → {config.OCR_DIR}/2022_narratives.json')


2015-2019 records : 3453
2022 records      : 1698
Combined total    : 5151

Records by year:
year
2015     666
2016     749
2017     456
2018     799
2019     783
2022    1698
Name: count, dtype: int64

Race (2015-2019):
race
Black      2132
White       830
Unknown     267
Asian         1
Name: count, dtype: int64

2022 with >=20-word narrative: 1697/1698

Saved → ../../datasets/02_ocr/police_narratives_all_years.json
2022 standalone → ../../datasets/02_ocr/2022_narratives.json


## 8. 2022 Quality Spot-Check

In [19]:
df_2022 = pd.DataFrame(records_2022)
df_2022['nar_words'] = df_2022['narrative_clean'].fillna('').apply(lambda x: len(str(x).split()))

print('=== 2022 Narrative Quality ===')
print(df_2022['nar_words'].describe().round(1))
print(f'< 10 words : {(df_2022["nar_words"] < 10).sum()}')
print(f'10-50 words: {((df_2022["nar_words"]>=10) & (df_2022["nar_words"]<50)).sum()}')
print(f'>= 50 words: {(df_2022["nar_words"] >= 50).sum()}')
print('\nOCR method breakdown:')
print(df_2022['ocr_model'].value_counts())

good = df_2022[df_2022['nar_words'] >= 50]
if len(good):
    print('\n=== Sample Narratives ===')
    for _, row in good.sample(min(3, len(good)), random_state=42).iterrows():
        print(f'\n[{row["cr_number"]}] {row["incident_date"]}  @  {row["incident_location"]}')
        print(row['narrative_clean'][:300])


=== 2022 Narrative Quality ===
count    1698.0
mean      258.3
std       144.7
min         0.0
25%       159.2
50%       231.5
75%       329.0
max      1268.0
Name: nar_words, dtype: float64
< 10 words : 1
10-50 words: 43
>= 50 words: 1654

OCR method breakdown:
ocr_model
pypdf_native                 1540
pypdf_native+gemini_parse     158
Name: count, dtype: int64

=== Sample Narratives ===

[2022-12-0090] Dec 31, 2022  @  190 Laburnam Crescent  Rochester NY 14620 -
On 12/31/2022 at 0431, I responded to the area of 226 Laburnam Cres for the report of 4 gun shots heard in the area. While enrt to the location, additional information was given to officers which included a male running towards Monroe Ave wearing tank top and grey sweatpants. Prior to arriving on sc

[2022-01-0067] Jan 19, 2022  @  72 Cuba Place  Rochester NY 14605 - Clinton
ON ABOVE DATE AND TIME I RESPONDED TO 72 CUBA PL FOR THE REPORT OF A DOMESTIC. UPON ARRIVAL I CAME ACROSS AN ERRATIC FEMALE (P) JENNIFER RODRIGUEZ. (P)